## 1) Instalare dependinte

Aceasta celula pregateste mediul pentru etapa de testare locala din server. Instaleaza `aiohttp` pentru WebSocket si `nest_asyncio` ca notebook-ul sa poata rula cod asincron fara blocaje.

In [ ]:
# Install the required packages if they are not available:
%pip install aiohttp nest_asyncio

## 2) Setup server si utilitare de baza

Aici se incarca importurile, se defineste portul serverului si se pregatesc functiile ajutatoare care gestioneaza lista de clienti. `_next_role()` alege cine primeste rolul urmator, `_peer_for()` gaseste celalalt client din pereche, iar `_send()` si `_relay()` trimit mesajele JSON mai departe.

In [25]:
import json
from aiohttp import WSMsgType, web
import nest_asyncio
from contextlib import redirect_stdout
from io import StringIO

In [26]:
nest_asyncio.apply()
PORT = 10001
print("[SERVER] Starting simple WebSocket pairer...")

_old_runner = globals().get("_runner")
if _old_runner is not None:
    try:
        await _old_runner.cleanup()
    except Exception as cleanup_error:
        print(f"[SERVER] Previous server cleanup failed: {cleanup_error}")

globals()["_runner"] = None
globals()["_site"] = None

app = web.Application()
# client = {"role": role, "ws": ws}
clients = []

# Aceasta functie primeste ca parametru websocket-ul unui client, si returneaza websocket-ul celuilalt client (peer-ul sau)
# sau None daca nu exista un peer conectat
def _peer_for(ws): 
    for item in clients:
        if item["ws"] is not ws:
            return item["ws"]
    return None


def _role_in_use(role):
    return any(item["role"] == role and not item["ws"].closed for item in clients)

# TODO1: 
# Implementati functia _next_role care returneaza care este URMATORUL ROL DISPONIBIL 
# adica "client1", "client2" sau None daca nu mai este niciunul

def _next_role():
    # if not _role_in_use("client1"):
    #     return "client1"
    # if not _role_in_use("client2"):
    #     return "client2"
    # return None
    pass

# HINT: Functia _role_in_use("clientX") returneaza True daca "clientX" este deja conectat, si False daca nu este conectat

async def _send(ws, payload):
    if not ws.closed:
        await ws.send_json(payload)

async def _relay(ws, payload):
    peer = _peer_for(ws)
    if peer is not None:
        await _send(peer, payload)


[SERVER] Starting simple WebSocket pairer...


### Verificare implementare

Mai jos aveți un **checker** care vă ajută să verificați dacă ce ați implementat mai sus este corect. Dacă implementarea este bună, checker-ul va afișa `PASSED`; altfel, va afișa `FAILED`.

In [30]:

_backup = clients[:]
W = lambda c=False: type("W", (), {"closed": c})()
try:
    clients[:] = []; a = (_next_role() == "client1")
    clients[:] = [{"role": "client1", "ws": W()}]; b = (_next_role() == "client2")
    clients[:] = [{"role": "client1", "ws": W()}, {"role": "client2", "ws": W()}]; c = (_next_role() is None)
    print("PASSED" if (a and b and c) else "FAILED")
finally:
    clients[:] = _backup

FAILED


## 3) Handler WebSocket si logica de semnalizare

Aceasta celula deschide conexiunea WebSocket pentru fiecare client si trimite mai departe mesajele de semnalizare WebRTC. WebSocket este canalul persistent prin care serverul poate primi si trimite mesaje in ambele directii fara sa redeschida conexiunea la fiecare pas.

`async` marcheaza functii care pot astepta operatii lente de retea fara sa blocheze notebook-ul, iar `await` spune codului sa astepte rezultatul acelui pas. Asta este important cand serverul primeste un mesaj, il trimite mai departe sau asteapta inchiderea unei conexiuni.

`_peer_for()` intoarce celalalt socket din pereche, iar serverul nu transporta video-ul sau sunetul propriu-zis. El muta doar mesajele de tip `offer`, `answer` si `ICE` intre cei doi clienti.

In [ ]:
async def websocket_handler(request):
    ws = web.WebSocketResponse(heartbeat=30)
    await ws.prepare(request)

    role = _next_role() # Aici am folosit functia implementata de voi mai sus
    if role is None:
        await _send(ws, {"type": "full", "message": "Room is full. Only two clients are allowed."})
        await ws.close()
        return ws

    #clients este lista de clienti conectati, unde fiecare client este un dictionar cu doua campuri
    # "role" - string si "ws" - obiectul WebSocketResponse asociat clientului respectiv

    #TODO2: 
    # Creati si adaugati clientul in lista de clienti conectati (stim deja ws si role)
    # Apoi, creati o variabila peer care reprezinta websocket-ul peer-ului sau
    # clients.append({"role": role, "ws": ws})
    # peer = _peer_for(ws)

    # HINT: Recititi cu atentie comentariile din celula precedenta


    print(f"[CONNECT] {role}")

    await _send(
        ws,
        {
            "type": "role",
            "role": role,
            "peer_role": "client2" if role == "client1" else "client1",
            "message": "Waiting for the other client..." if peer is None else "Both clients are connected.",
        },
    )

    if peer is not None:
        await _send(peer, {"type": "peer_connected", "message": "The other client joined."})

    try:
        async for msg in ws:
            if msg.type != WSMsgType.TEXT:
                continue

            try:
                payload = json.loads(msg.data)
            except json.JSONDecodeError:
                await _send(ws, {"type": "error", "message": "Invalid JSON payload."})
                continue

            msg_type = payload.get("type")
            if msg_type in {"webrtc_offer", "webrtc_answer", "webrtc_ice_candidate"}:
                await _relay(ws, {"type": msg_type, "data": payload.get("data")})
            else:
                await _send(ws, {"type": "error", "message": f"Unknown message type: {msg_type}"})
    finally:
        peer = _peer_for(ws)
        clients[:] = [item for item in clients if item["ws"] is not ws]
        print(f"[DISCONNECT] {role}")
        if peer is not None and not peer.closed:
            await _send(peer, {"type": "peer_disconnected", "message": "The other client disconnected."})

    return ws


app.router.add_get("/ws", websocket_handler)


<ResourceRoute [GET] <PlainResource  /ws> -> <function websocket_handler at 0x110aebe20>

## 4) Pornire si oprire server

Aici se porneste `AppRunner`/`TCPSite` si se pregateste functia de cleanup. Serverul ramane activ dupa ce celula se termina, fiindca asculta in fundal in event loop-ul notebook-ului; celula nu tine video sau audio, ci doar lasa serverul pornit.

In [ ]:
_runner = globals().get("_runner")
_site = globals().get("_site")


async def _start_server(port=PORT):
    global _runner, _site

    # If an old runner reference exists, clean it up and start fresh.
    if _runner is not None:
        try:
            await _runner.cleanup()
        except Exception:
            pass
        _runner = None
        _site = None

    _runner = web.AppRunner(app)
    await _runner.setup()
    _site = web.TCPSite(_runner, "127.0.0.1", port)

    try:
        await _site.start()
        print("===================================================")
        print(f" SIGNALING SERVER RUNNING ON PORT {port}")
        print(f" WebSocket endpoint: ws://127.0.0.1:{port}/ws")
        print("===================================================")
    except OSError as exc:
        await _runner.cleanup()
        _runner = None
        _site = None
        if getattr(exc, "errno", None) == 48:
            print(f"[ERROR] Port {port} is already in use.")
            return
        raise

# TODO3: Implementati functia de stop_server ce opreste serverul curent
# daca serverul nu ruleaza, afisati: "[SERVER] Not running." si iesiti din functie
# altfel apelati functia de cleanup() a runner-ului asa cum a fost folosita de mai multe ori mai sus in cod
# resetati variabilele globale pe None si goliti si lista de clienti conectati creata anterior de voi
# in acest caz (daca am avut ceva sa oprim) afisati un mesaj de confirmare: "[SERVER] Stopped."
async def stop_server():
    global _runner, _site

    # if _runner is None:
    #     print("[SERVER] Not running.")
    #     return

    # await _runner.cleanup()
    # _runner = None
    # _site = None
    # clients.clear()
    # print("[SERVER] Stopped.")

# HINT: Aveti un checker mai jos pentru a verifica daca implementarea voastra este corecta 



### Verificare implementare

Mai jos aveți un **checker** care vă ajută să verificați dacă implementarea de mai sus este corectă. Dacă totul este în regulă, checker-ul va afișa `PASSED`; altfel, va afișa `FAILED`.

In [ ]:
_runner0, _site0, clients0 = _runner, _site, clients[:]
class R:
    async def cleanup(self): pass
_runner, _site, clients[:] = R(), object(), [1]
with redirect_stdout(StringIO()): await stop_server()
print("PASSED" if (_runner is None and _site is None and not clients) else "FAILED")
_runner, _site, clients[:] = _runner0, _site0, clients0

In [ ]:
# Ce se intampla aici?
await _start_server(port=PORT)

# A) se creaza doar obiectul serverului, urmand sa fie pornit ulterior cand cei 2 clienti se vor conecta
# B) se porneste serverul pe portul din PORT, iar daca portul e ocupat asteapta sa se elibereze portul
# C) se porneste serverul daca portul este liber, iar daca portul este ocupat se afiseaza un mesaj de eroare
# D) se seteaza portul serverului la valoarea din PORT, nu are legatura cu pornirea efectiva a serverului

# Raspuns: 


Oprirea manuala a serverului, dupa ce am testat:
Aceasta celula opreste backend-ul de signaling si elibereaza portul. Conexiunile audio si video ale clientilor nu se opresc de aici, pentru ca ele sunt peer-to-peer si nu trec prin server.

In [31]:
await stop_server()

[SERVER] Not running.


## 5) TODO 4: Blur camera si noise suppression

Aceasta este urmatoarea extensie pentru frontend: un buton de blur pentru camera si un buton de noise suppression pentru voce. Deocamdata butoanele exista, dar sunt dezactivate si nu fac nimic pana cand studentii implementeaza efectiv logica.

Checker-ul de mai jos arata ce trebuie sa existe in final: butoanele trebuie sa fie prezente, active si conectate la logica reala, iar placeholder-ele trebuie eliminate. Pana atunci, el va arata FAIL pe partea de implementare, ceea ce este normal pentru un TODO.

In [20]:
# Aici o sa trebuaisca sa implmenteze functia pentru blur/deblur de camera
# si functia pentru noise supression
#blur camera și noise suppression în frontend, lăsate momentan inactive, plus un checker simplu 
# care verifică existența butoanelor, starea lor și dacă placeholder-ele au fost scoase. 
# Checker-ul este făcut intenționat să arate FAIL până când studenții implementează efectiv 
# funcționalitatea.


In [21]:
# dupa care ultima si cea mai grea parte care are mai multe chestii de facut dar e o singura cerinta: 
# sa copieze de mai sus toate celulele si sa schimbe fiecare astfel incat sa se conecteze 2 persoane
# de pe pcuri diferite la un call

# va treebui sa schimbe in client1 si doar client1 va fi rulat 
# serverul va trebui schimbat si el practic il fac de la 0 avand codul pentru cum faci sa intri de pe doua taburi diferite


Pe scurt: nu, nu trebuie să copieze tot serverul, dar pentru două PC-uri nu ajunge doar Server.ipynb dacă web/main.js rămâne fix pe localhost. Serverul din notebook deja face partea grea de signaling și pairing, iar rolurile client1/client2 se dau automat la conectare, deci nu ai nevoie de butoane separate pentru ele. În practică, utilizatorii doar deschid pagina și apasă Join; primul devine client1, al doilea client2.

Ce blochează două mașini diferite este faptul că browserul trebuie să știe IP-ul mașinii care rulează serverul de signaling. Dacă acel URL rămâne ws://127.0.0.1:10001/ws, al doilea PC va încerca să se conecteze la el însuși și va pica. Deci, dacă nu vrei să atingi deloc partea de web, atunci nu e suficient doar notebook-ul; trebuie măcar un mecanism prin care clientul primește adresa serverului. Se poate face fie cu o mică adaptare în client, fie cu o celulă din notebook care generează configurarea automat, dar numai codul din notebook, fără niciun fel de configurare client-side, nu va fi destul pentru două PC-uri.
